# Generacion de Queries Sinteticas para Dataset LTR
## DSRP - Curso de Ingenieria de ML

Este notebook genera queries sinteticas diversas usando:
- **Ollama (Llama 3.2:3b)** para generacion creativa de queries
- **Plantillas basadas en reglas** como queries de linea base

El objetivo es crear un dataset de Learning-to-Rank (LTR) para entrenar modelos de recomendacion.

**Prerequisito**: Ejecutar primero `feature_engineering.ipynb` para generar:
- `data/complete_imdb_database.parquet`
- `data/movie_embs.npy`

In [ ]:
import requests
import numpy as np
import polars as pl
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Utilidades compartidas
from ml_utils import (
    extract_top_genres,
    extract_decades,
    generate_template_queries,
    normalize_embeddings,
    get_candidates_for_query,
    compute_relevance_score,
    assign_relevance_labels,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_TOP_K_CANDIDATES,
    DEFAULT_N_LABEL_BINS,
)

## 1. Configuracion

In [ ]:
# Configuracion de Ollama
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:3b"

# Rutas de datos
DATA_PATH = "data/complete_imdb_database.parquet"
EMBEDDINGS_PATH = "data/movie_embs.npy"
OUTPUT_PATH = "data/ltr_imdb_dataset.parquet"

# Parametros LTR
TOP_K_CANDIDATES = DEFAULT_TOP_K_CANDIDATES
N_LABEL_BINS = DEFAULT_N_LABEL_BINS

## 2. Cargar Datos y Modelos

In [ ]:
# Cargar base de datos de peliculas
movies_df = pl.read_parquet(DATA_PATH)
print(f"Se cargaron {movies_df.height} peliculas")
print(f"Columnas: {movies_df.columns}")

# Agregar imdb_votes_log si no existe
if "imdb_votes_log" not in movies_df.columns:
    movies_df = movies_df.with_columns(
        pl.col("imdb_votes").log1p().alias("imdb_votes_log")
    )

movies_df.head(3)

In [ ]:
# Cargar y normalizar embeddings
movie_embs = np.load(EMBEDDINGS_PATH).astype("float32")
movie_embs_norm = normalize_embeddings(movie_embs)
print(f"Dimensiones de embeddings: {movie_embs.shape}")

In [ ]:
# Cargar modelo de embeddings
model = SentenceTransformer(DEFAULT_EMBEDDING_MODEL)
print(f"Modelo cargado: {DEFAULT_EMBEDDING_MODEL}")

## 3. Extraer Metadata para Generacion de Queries

In [ ]:
# Extraer generos y decadas
top_genres = extract_top_genres(movies_df, top_n=15)
decades = extract_decades(movies_df)

print(f"Generos principales: {top_genres}")
print(f"Decadas: {decades}")

In [ ]:
# Seleccionar peliculas populares para contexto del LLM
sample_movies = (
    movies_df
    .filter(pl.col("imdb_votes") > 50000)
    .sort("imdb_rating", descending=True)
    .head(50)
    .select(["title", "genres", "year"])
)

sample_titles = sample_movies["title"].to_list()[:20]
print(f"Peliculas populares de muestra: {sample_titles[:5]}")

## 4. Generacion de Queries con Ollama

Usar Llama 3.2 para generar queries de busqueda de peliculas diversas y en lenguaje natural.

In [ ]:
def query_ollama(prompt: str, model: str = OLLAMA_MODEL) -> str:
    """Consultar API de Ollama y retornar texto generado."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.8,
            "top_p": 0.9,
            "num_predict": 200,
        }
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=30)
        response.raise_for_status()
        return response.json().get("response", "")
    except requests.exceptions.RequestException as e:
        print(f"Error de Ollama: {e}")
        return ""


def parse_queries_from_response(response: str) -> list[str]:
    """Parsear lista numerada de queries de la respuesta del LLM."""
    queries = []
    for line in response.strip().split("\n"):
        line = line.strip()
        if line and line[0].isdigit():
            parts = line.split(".", 1) if "." in line[:3] else line.split(")", 1)
            if len(parts) > 1:
                line = parts[1].strip()
        elif line.startswith("- "):
            line = line[2:].strip()
        
        line = line.strip("\"'")
        if line and len(line) > 5 and len(line) < 100:
            queries.append(line.lower())
    
    return queries

In [ ]:
# Probar conexion con Ollama
test_response = query_ollama("Say 'Ollama is working' in one line.")
ollama_available = bool(test_response)
print(f"Ollama disponible: {ollama_available}")
if test_response:
    print(f"Respuesta: {test_response}")

In [ ]:
def generate_llm_queries(category: str, context: str, num_queries: int = 5) -> list[dict]:
    """Generar queries diversas usando LLM para una categoria especifica."""
    
    prompt = f"""You are helping create a movie search dataset. Generate {num_queries} diverse, natural movie search queries for: {category}

Context: {context}

Requirements:
- Each query should be how a real user would search for movies
- Vary the phrasing (some formal, some casual)
- Include different intents: browsing, specific mood, recommendations
- Keep queries between 3-12 words
- Output ONLY the queries as a numbered list, nothing else

Generate {num_queries} queries:"""

    response = query_ollama(prompt)
    queries = parse_queries_from_response(response)
    
    return [
        {
            "query_text": q,
            "intent_type": "llm_generated",
            "category": category,
            "emphasis": "neutral",
            "genre": None,
            "decade": None,
        }
        for q in queries[:num_queries]
    ]

In [ ]:
llm_queries = []

if ollama_available:
    # Queries basadas en genero
    print("Generando queries basadas en genero...")
    for genre in tqdm(top_genres[:8]):
        queries = generate_llm_queries(
            category=f"{genre} movies",
            context=f"Genre: {genre}. Popular examples exist in our database.",
            num_queries=5
        )
        for q in queries:
            q["genre"] = genre
        llm_queries.extend(queries)
    
    # Queries basadas en estado de animo
    print("Generando queries basadas en estado de animo...")
    moods = [
        ("relaxing weekend", "Movies for a relaxing weekend"),
        ("date night", "Romantic or engaging movies for couples"),
        ("family movie night", "Family-friendly movies"),
        ("mind-bending", "Complex, thought-provoking films"),
        ("adrenaline rush", "Action-packed, exciting movies"),
        ("hidden gems", "Underrated quality films"),
    ]
    
    for mood, context in tqdm(moods):
        queries = generate_llm_queries(category=mood, context=context, num_queries=5)
        llm_queries.extend(queries)
    
    # Queries de similitud
    print("Generando queries de similitud...")
    for title in tqdm(sample_titles[:8]):
        queries = generate_llm_queries(
            category=f"movies similar to {title}",
            context=f"Reference movie: {title}",
            num_queries=3
        )
        for q in queries:
            q["intent_type"] = "similarity_search"
        llm_queries.extend(queries)

print(f"Total queries LLM: {len(llm_queries)}")

## 5. Queries Basadas en Plantillas

Generar queries estructuradas usando plantillas para cobertura sistematica.

In [ ]:
# Generar queries usando plantillas
template_queries = generate_template_queries(top_genres, decades)
print(f"Se generaron {len(template_queries)} queries de plantilla")

## 6. Combinar y Deduplicar Queries

In [ ]:
# Combinar todas las queries
all_queries = llm_queries + template_queries

# Deduplicar por query_text
seen = set()
unique_queries = []
for q in all_queries:
    text = q["query_text"].lower().strip()
    if text not in seen:
        seen.add(text)
        unique_queries.append(q)

# Asignar IDs de query
for i, q in enumerate(unique_queries, start=1):
    q["query_id"] = i

# Crear DataFrame
queries_df = pl.DataFrame(
    unique_queries,
    schema={
        "query_text": pl.Utf8,
        "intent_type": pl.Utf8,
        "category": pl.Utf8,
        "emphasis": pl.Utf8,
        "genre": pl.Utf8,
        "decade": pl.Int64,
        "query_id": pl.Int64,
    }
)

print(f"Total queries unicas: {queries_df.height}")
queries_df.head(10)

In [ ]:
# Distribucion de queries
print("\nQueries por tipo de intencion:")
print(queries_df.group_by("intent_type").len().sort("len", descending=True))

print("\nQueries por enfasis:")
print(queries_df.group_by("emphasis").len().sort("len", descending=True))

## 7. Recuperacion de Candidatos

Para cada query, recuperar las top-K peliculas candidatas usando similitud de embeddings.

In [ ]:
print(f"Recuperando top-{TOP_K_CANDIDATES} candidatos para {queries_df.height} queries...")

all_candidates = []
for q_row in tqdm(queries_df.iter_rows(named=True), total=queries_df.height):
    cand = get_candidates_for_query(
        query_id=q_row["query_id"],
        query_text=q_row["query_text"],
        movies_df=movies_df,
        movie_embs_norm=movie_embs_norm,
        model=model,
        k=TOP_K_CANDIDATES,
    )
    all_candidates.append(cand)

candidates_df = pl.concat(all_candidates)
print(f"\nTotal candidatos: {candidates_df.height}")
candidates_df.head(5)

## 8. Scoring de Relevancia y Etiquetado

Calcular scores de relevancia basados en:
- Similitud de embeddings
- Rating de IMDB
- Popularidad (votos)

Los pesos varian segun el enfasis de la query.

In [ ]:
print("Calculando scores de relevancia y etiquetas...")

queries_by_id = {row["query_id"]: row for row in queries_df.iter_rows(named=True)}
ltr_chunks = []

for qid, q_row in tqdm(queries_by_id.items(), total=len(queries_by_id)):
    cand = candidates_df.filter(pl.col("query_id") == qid)
    if cand.is_empty():
        continue

    # Agregar score de relevancia
    cand = compute_relevance_score(cand, emphasis=q_row.get("emphasis", "neutral"))

    # Agregar etiquetas discretas
    cand = assign_relevance_labels(cand, n_bins=N_LABEL_BINS)

    ltr_chunks.append(cand)

ltr_df = pl.concat(ltr_chunks)
print(f"\nTamano del dataset LTR: {ltr_df.height}")

## 9. Dataset Final

In [ ]:
print(f"Forma del dataset LTR final: {ltr_df.shape}")
print(f"\nColumnas: {ltr_df.columns}")
ltr_df.head(20)

In [ ]:
# Distribucion de etiquetas
print("\nDistribucion de etiquetas:")
print(ltr_df.group_by("label").len().sort("label"))

In [ ]:
# Muestra de una query
sample_qid = ltr_df["query_id"].unique()[0]
sample_query = ltr_df.filter(pl.col("query_id") == sample_qid)

print(f"Query de muestra: '{sample_query['query_text'][0]}'")
print(f"\nTop 10 resultados:")
sample_query.head(10)

In [ ]:
# Guardar dataset final
ltr_df.write_parquet(OUTPUT_PATH)
print(f"Dataset LTR guardado en {OUTPUT_PATH}")

## 10. Resumen

In [ ]:
print("=" * 50)
print("RESUMEN DEL DATASET LTR")
print("=" * 50)
print(f"Total de queries: {ltr_df['query_id'].n_unique()}")
print(f"Total de pares query-documento: {ltr_df.height}")
print(f"Candidatos por query: {TOP_K_CANDIDATES}")
print(f"Bins de etiquetas: {N_LABEL_BINS} (0-{N_LABEL_BINS-1})")
print(f"\nArchivo generado: {OUTPUT_PATH}")
print(f"\nSiguiente paso: Ejecutar modeling.ipynb para entrenar el modelo LTR")